# Hermes — VN30F Feature Engineering & EDA

**Scope**: Research prototype only — Hermes Quant Analyst workspace.  
**Goal**: Load VN30F OHLCV data, compute technical + microstructure features, and perform exploratory data analysis.

> ⚠️ This notebook is for research/hypothesis validation only. No production trading logic is implemented here.

In [ ]:
import sys, pathlib
# Ensure the repo root is on the path
ROOT = pathlib.Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from research.data.loader import load_ohlcv
from research.features.technical import add_returns, add_volatility, add_momentum
from research.features.microstructure import add_spread_proxy, add_amihud_illiquidity

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Imports OK')

## 1. Load Data

Replace `DATA_PATH` with the path to your VN30F CSV file.  
Required columns: `datetime, open, high, low, close, volume`

In [ ]:
DATA_PATH = '../data/vn30f_1d.csv'  # adjust path

try:
    df = load_ohlcv(DATA_PATH)
    print(f'Loaded {len(df):,} bars from {df.index[0].date()} to {df.index[-1].date()}')
    display(df.head())
except FileNotFoundError:
    print(f'[INFO] Data file not found at {DATA_PATH}.')
    print('Generating synthetic VN30F-like data for demonstration...')
    np.random.seed(42)
    n = 500
    idx = pd.bdate_range('2022-01-01', periods=n, tz='Asia/Ho_Chi_Minh')
    close = 1200 + np.cumsum(np.random.randn(n) * 8)
    df = pd.DataFrame({
        'open':   close - np.abs(np.random.randn(n) * 3),
        'high':   close + np.abs(np.random.randn(n) * 5),
        'low':    close - np.abs(np.random.randn(n) * 5),
        'close':  close,
        'volume': np.random.randint(5000, 30000, n),
    }, index=idx)
    print(f'Synthetic data: {len(df):,} bars')

## 2. Feature Engineering

In [ ]:
df = add_returns(df, periods=[1, 5, 20])
df = add_volatility(df, windows=[5, 20, 60])
df = add_momentum(df, windows=[5, 20, 60])
df = add_spread_proxy(df, window=20)
df = add_amihud_illiquidity(df, window=20)

print(f'Feature columns: {df.columns.tolist()}')
display(df.tail())

## 3. Price & Volatility Overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df.index, df['close'], lw=1, label='Close')
axes[0].set_ylabel('Price (VND pts)')
axes[0].set_title('VN30F Close Price')
axes[0].legend()

axes[1].fill_between(df.index, df['vol_20'], alpha=0.5, label='RV 20-bar')
axes[1].fill_between(df.index, df['vol_60'], alpha=0.3, label='RV 60-bar')
axes[1].set_ylabel('Annualised Vol')
axes[1].set_title('Realised Volatility')
axes[1].legend()

axes[2].bar(df.index, df['volume'], width=0.8, label='Volume')
axes[2].set_ylabel('Volume (contracts)')
axes[2].set_title('Daily Volume')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Feature Correlation

In [ ]:
feat_cols = [c for c in df.columns if c not in ('open', 'high', 'low', 'close', 'volume')]
corr = df[feat_cols].dropna().corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Return Distribution Analysis

In [ ]:
from scipy import stats

ret = df['ret_1'].dropna()
print(f'Observations  : {len(ret):,}')
print(f'Mean          : {ret.mean():.6f}')
print(f'Std           : {ret.std():.6f}')
print(f'Skewness      : {ret.skew():.3f}')
print(f'Excess Kurtosis: {ret.kurtosis():.3f}')

stat, p = stats.jarque_bera(ret)
print(f'Jarque-Bera   : stat={stat:.2f}, p={p:.4f} ({"non-normal" if p < 0.05 else "normal"})')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(ret, bins=50, density=True, alpha=0.7)
x = np.linspace(ret.min(), ret.max(), 300)
axes[0].plot(x, stats.norm.pdf(x, ret.mean(), ret.std()), 'r--', label='Normal fit')
axes[0].set_title('Return Distribution')
axes[0].legend()

stats.probplot(ret, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot')
plt.tight_layout()
plt.show()